# 04 ? Bayesian hyperparameter optimisation

Optuna TPE tunes the HistGradientBoosting baseline winner. Evaluation uses five GroupKFold splits over training trips only. To keep runtime bounded, each fold uses a deterministic, balanced training sample (up to 5,000 rows per class) and a stratified validation sample (up to 20,000 rows). Unknown labels (-1) are excluded. Student-test and evaluator-test data are not used for tuning.


In [ ]:
from pathlib import Path
import json, time
import numpy as np
import pandas as pd
import optuna
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import GroupKFold, train_test_split
from sklearn.metrics import f1_score

TRAIN_CSV = Path("data/processed/train_candidates.csv")
MANIFEST = Path("configs/person_b_feature_manifest.json")
BASELINE = Path("experiments/results/person_b/baseline_model_comparison.csv")
OUT = Path("experiments/results/person_b")
OUT.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(TRAIN_CSV)
manifest = json.loads(MANIFEST.read_text(encoding="utf-8"))
features = manifest["feature_columns"]
df = df[df["is_match"].isin([0, 1])].copy()
df["is_match"] = df["is_match"].astype("int8")
summary = pd.read_csv(BASELINE, index_col=0)
family = summary["macro_f1_mean"].idxmax()
if family != "hist_gradient_boosting":
    raise RuntimeError(f"Expected verified baseline winner HistGradientBoosting, got {family}")
X = df[features].astype("float32")
y = df["is_match"]
groups = df["trip_id"]
print("Tuning family:", family)
print("Labelled rows:", len(df), "Trips:", groups.nunique(), "Features:", features)
print("Class counts:", y.value_counts().to_dict())


In [ ]:
SEED = 42
N_TRIALS = 12
cv = GroupKFold(n_splits=5)

def sample_indices(indices, labels, cap_per_class, seed):
    rng = np.random.default_rng(seed)
    chosen = []
    vals = labels[indices]
    for cls in (0, 1):
        pool = indices[vals == cls]
        if len(pool) > cap_per_class:
            pool = rng.choice(pool, cap_per_class, replace=False)
        chosen.append(pool)
    return np.concatenate(chosen)

y_np = y.to_numpy()
fold_data = []
for fold, (tr_all, va_all) in enumerate(cv.split(X, y, groups), 1):
    tr = sample_indices(tr_all, y_np, 5000, SEED + fold)
    # Stratified validation sample, independent of model/trial.
    if len(va_all) > 20000:
        va, _ = train_test_split(
            va_all, train_size=20000, stratify=y_np[va_all],
            random_state=SEED + 100 + fold
        )
    else:
        va = va_all
    train_trips = set(groups.iloc[tr])
    valid_trips = set(groups.iloc[va])
    if train_trips.intersection(valid_trips):
        raise RuntimeError("Trip leakage detected")
    fold_data.append((tr, va))
    print(f"Fold {fold}: train={len(tr):,}, validation={len(va):,}, trips={len(train_trips)}/{len(valid_trips)}")


In [ ]:
def objective(trial):
    params = {
        "learning_rate": trial.suggest_float("learning_rate", 0.02, 0.2, log=True),
        "max_iter": trial.suggest_int("max_iter", 75, 250, step=25),
        "max_leaf_nodes": trial.suggest_int("max_leaf_nodes", 15, 63),
        "l2_regularization": trial.suggest_float("l2_regularization", 1e-4, 10.0, log=True),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 10, 60, step=5),
        "random_state": SEED
    }
    scores = []
    for fold, (tr, va) in enumerate(fold_data, 1):
        model = HistGradientBoostingClassifier(**params)
        model.fit(X.iloc[tr], y.iloc[tr])
        pred = model.predict(X.iloc[va])
        scores.append(f1_score(y.iloc[va], pred, average="macro", zero_division=0))
    trial.set_user_attr("fold_scores", [float(s) for s in scores])
    return float(np.mean(scores))

# In-memory storage avoids the blocked SQLAlchemy native extension.
# Every trial is exported to CSV and the best configuration to JSON.
study = optuna.create_study(
    study_name="pathsnap_person_b_tpe_fastfix",
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=SEED)
)
remaining = max(0, N_TRIALS - len(study.trials))
print("Existing trials:", len(study.trials), "New trials:", remaining)
start = time.perf_counter()
if remaining:
    study.optimize(objective, n_trials=remaining, show_progress_bar=False)
elapsed = time.perf_counter() - start

trials = study.trials_dataframe()
trials.to_csv(OUT / "bayesian_optimization_trials.csv", index=False)
best = {
    "baseline_family": "hist_gradient_boosting",
    "best_value_macro_f1_groupkfold": float(study.best_value),
    "best_params": study.best_params,
    "n_trials_total": len(study.trials),
    "optimisation_seconds_this_run": elapsed,
    "sampler": "Optuna TPESampler",
    "objective": "mean sampled GroupKFold macro-F1 on training trips only",
    "folds": 5,
    "training_sample_per_class_per_fold": 5000,
    "validation_sample_per_fold": 20000,
    "student_test_used_for_tuning": False,
    "evaluator_test_used": False
}
(OUT / "bayesian_optimization_best.json").write_text(json.dumps(best, indent=2), encoding="utf-8")
print(json.dumps(best, indent=2))


In [ ]:
complete = trials[trials["state"] == "COMPLETE"].copy()
if complete.empty:
    raise RuntimeError("No completed Optuna trials")
plt.figure(figsize=(9, 4))
plt.plot(np.arange(1, len(complete) + 1), complete["value"].cummax(), marker="o", markersize=3)
plt.xlabel("Completed trial")
plt.ylabel("Best-so-far sampled GroupKFold macro-F1")
plt.title("Optuna TPE optimisation history")
plt.grid(alpha=0.25)
plt.tight_layout()
plot_path = OUT / "bayesian_optimization_history.png"
plt.savefig(plot_path, dpi=160, bbox_inches="tight")
plt.close()
print("Saved:", plot_path)
